In [1]:
# model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/baseline_passk_training/actor/global_step_350"
# model_path = "/mnt/petrelfs/share_data/zhangshilin/entropy_clip_cov"
model_path = "/mnt/petrelfs/share_data/zhangshilin/rs_equ_100step"
# model_path = "/mnt/petrelfs/share_data/zhangshilin/baseline_clp_02_028_300"

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [3]:

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.85,
    dtype="auto"
)

INFO 09-22 10:53:43 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 09-22 10:53:43 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/share_data/zhangshilin/rs_equ_100step', speculative_config=None, tokenizer='/mnt/petrelfs/share_data/zhangshilin/rs_equ_100step', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name=/mnt/petrelfs/share_data/zhangshilin/rs_equ_100step, use_v2_block_manager=False, enable

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 09-22 10:53:57 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 09-22 10:53:59 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 09-22 10:54:01 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-22 10:54:01 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-22 10:54:17 model_runner.py:1225] Graph capturing finished in 16 secs.


In [22]:
val_data_path = "dataset/eval.passn.parquet"
val_dataset = RLHFDataset(parquet_files=val_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=128,
                            shuffle=False,
                            drop_last=False,
                            collate_fn=collate_fn)
n_val_samples = 8

original dataset len: 1590
filter dataset len: 1588


In [32]:
data_idx = 1000
test_data = val_dataset[data_idx]
# print(test_data['reward_model'])
input_text = tokenizer.decode(test_data['input_ids'], skip_special_tokens=True)
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)



In [30]:
prompts = [input_text] * n_val_samples
outputs = base_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 8/8 [00:29<00:00,  3.66s/it, est. speed input: 76.19 toks/s, output: 482.82 toks/s]


In [31]:
input_text
test_data

{'data_source': 'olympiad_bench',
 'ability': 'math',
 'reward_model': {'ground_truth': '12', 'style': 'rule'},
 'extra_info': {'index': 502, 'split': 'default'},
 'input_ids': tensor([151643, 151643, 151643,  ...,    366,  26865,    397]),
 'attention_mask': tensor([0, 0, 0,  ..., 1, 1, 1]),
 'position_ids': tensor([  0,   0,   0,  ..., 276, 277, 278]),
 'index': 502}

In [33]:
group_rollout = []
for output in outputs:
    # 提取生成的文本
    full_text = output.outputs[0].text
    # 只保留输入之后新生成的部分
    generated_text = full_text[len(input_text):]
    group_rollout.append(generated_text)

In [34]:
print(test_data['reward_model'])
for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])  # 从列表中提取文本
    print("__________end_____________\n")

{'ground_truth': '12', 'style': 'rule'}
********0**********
):

1. If \(m\) of the \(x_i\)’s are \(a\) and the remaining \(k-m\) are \(-a\), then:
\[
P = a^m (-a)^{k-m} = a^m (-1)^{k-m} a^{k-m} = a^k (-1)^{k-m}.
\]
Since \(P = a^2\), it follows that:
\[
a^k (-1)^{k-m} = a^2.
\]
If \(a \neq 0\), we can divide both sides by \(a^2\):
\[
a^{k-2} (-1)^{k-m} = 1.
\]
This equation holds true if and only if:
\[
a^{k-2} = \frac{1}{(-1)^{k-m}}.
\]
If \(a \neq 0\), \(a^{k-2} \neq 0\). For the equality \(a^{k-2} = \frac{1}{(-1)^{k-m}}\) to hold, \(\frac{1}{(-1)^{k-m}}\) must be a real number, implying \((-1)^{k-m} = \pm 1\). This implies \(k-m\) must be even because only then \((-1)^{\text{even number}} = 1\). Hence, \(k-m\) must be even.

2. If \(a = 0\), then \(P = 0\). Hence, \(x_1 x_2 \cdots x_k = 0\) implies at least one \(x_i\) is zero. If any \(x_i = 0\), all \(x_j\) must be zero (because \(x_j^2 = 0\) for all \(j\)). Thus, the \(k\)-tuple would be \((0, 0, \ldots, 0)\), which is a trivial 